# 資料視覺化：選對圖，讀懂資料

已學過 vibe coding，繪圖程式可交給 AI 協助；本篇重點是選圖與判讀。
只保留常用範例，不展開有限狀態機、配色、marker、樣式切換與進階排版。
套件安裝請見 [使用說明](./README.md)。

## 1. 先決定要回答什麼問題

|問題|圖表|判讀重點|
|---|---|---|
|數值如何隨時間變動？|折線圖|趨勢、突變、時間順序|
|數值集中在哪裡？|直方圖|分布、偏斜、可能的極端值|
|兩個特徵有關係嗎？|散佈圖|關聯、群聚、離群點；關聯不代表因果|
|各類別有幾筆？|長條圖|樣本數是否不均衡|
|數值特徵是否高度相關？|相關係數熱圖|可能重複的資訊；相關性不等於預測能力|

Matplotlib 負責一般繪圖，Seaborn 方便直接對表格繪製統計圖。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 隨附的模擬資料；從 simplified 目錄執行。
df = pd.read_csv('./data/weather_sample.csv')
print("缺失筆數：\n", df.isna().sum())

## 2. 趨勢與分布

氣溫缺失時，折線會中斷；直方圖忽略缺失值。
不為了讓圖連續就擅自填值。直方圖的 bins 會影響外觀，需配合樣本數解讀。
`bins=5` 代表自動分成 5 個區間，也可以傳入邊界清單，例如 `[15, 20, 25, 30]`；
比較不同組別時，使用相同邊界才方便比較。以下直方圖與多子圖整理自原繪圖教材。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(df['Day'], df['Temperature'], marker='o')
axes[0].set(xlabel='Day', ylabel='Temperature (C)', title='Temperature over time')
axes[1].hist(df['Temperature'].dropna(), bins=5)
axes[1].set(xlabel='Temperature (C)', ylabel='Count', title='Temperature distribution')
fig.tight_layout()
plt.show()

**看圖後回答：**哪些天氣溫較高？缺資料的是哪一天？只有 12 筆模擬資料，能否支持季節性結論？

## 3. 特徵關係與類別數量

散佈圖用顏色區分是否下雨；它能幫助提出假設，不能取代模型驗證。
長條圖呈現每一類的樣本數，先確認資料是否偏向某一類。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
sns.scatterplot(data=df, x='Temperature', y='Humidity', hue='Rain', ax=axes[0])
axes[0].set(title='Temperature vs humidity', xlabel='Temperature (C)', ylabel='Humidity (%)')
df['Rain'].value_counts().sort_index().plot.bar(ax=axes[1], rot=0)
axes[1].set(xlabel='Rain (0=no, 1=yes)', ylabel='Count', title='Class counts')
fig.tight_layout()
plt.show()

### 小練習 5：比較兩組氣溫分布

使用 `df`，將下雨與沒下雨的有效氣溫畫在同一張圖的兩個子圖中，兩組使用相同的
分箱邊界 `[15, 18, 21, 24, 27]`，標示各組名稱與座標，並印出各組有效樣本數。
根據圖與樣本數，說明這份資料是否足以推論一般天氣規律。

<details>
<summary>參考答案</summary>

```python
no_rain = df.loc[df['Rain'] == 0, 'Temperature'].dropna()
rain = df.loc[df['Rain'] == 1, 'Temperature'].dropna()
print('沒下雨有效樣本數：', len(no_rain))
print('下雨有效樣本數：', len(rain))
fig_ex, axes_ex = plt.subplots(1, 2, figsize=(10, 3))
axes_ex[0].hist(no_rain, bins=[15, 18, 21, 24, 27])
axes_ex[1].hist(rain, bins=[15, 18, 21, 24, 27])
axes_ex[0].set(xlabel='Temperature (C)', ylabel='Count', title='No rain')
axes_ex[1].set(xlabel='Temperature (C)', ylabel='Count', title='Rain')
fig_ex.tight_layout()
plt.show()
```

沒下雨組有 9 筆有效氣溫，下雨組只有 2 筆；這是少量模擬資料，無法據此推論一般天氣規律。

</details>

## 4. 相關係數熱圖

先用 `.corr()` 計算相關係數，再交給 `heatmap` 顯示；熱圖本身不會計算相關性。
相關係數接近 ±1 表示較強的線性關聯，接近 0 不代表完全沒有關係。
本例含缺失值，pandas 以各欄位配對後的有效資料計算。

In [ ]:
corr = df[['Temperature', 'Humidity']].corr()
sns.heatmap(corr, annot=True, vmin=-1, vmax=1, cmap='coolwarm')
plt.title('Feature correlation')
plt.tight_layout()
plt.show()

## 練習：讓 AI 畫圖，由你驗收

請 AI 對同一份資料比較下雨／沒下雨時的氣溫分布，並提出一個觀察。
可用提示：「以 `Rain` 分組畫 `Temperature` 箱型圖，標示單位與各組有效樣本數，說明缺失值處理方式。」

驗收：分組和欄位是否正確？各組有幾筆？極端值是否真的錯誤？
圖表的觀察是否超過這份小型模擬資料能支持的範圍？

In [ ]:
# 參考解答：僅繪製有氣溫記錄的樣本，未填補缺失值。
observed = df.dropna(subset=['Temperature'])
print("各組有效樣本數：\n", observed.groupby('Rain')['Temperature'].count())
sns.boxplot(data=observed, x='Rain', y='Temperature')
plt.xlabel('Rain (0=no, 1=yes)')
plt.ylabel('Temperature (C)')
plt.title('Temperature by rain category')
plt.show()